# **Datalab II Sprint 3 2025 - 2026**
**Plaats:** De Haagse Hogenschool, ADS & AI<br>
**Auteurs:** M. Kilinc, D. Hoogenbosch, J. Wolthuis, S. Slingerland, L. van Hamersveld<br>
**Groep:** B2 <br>
**Coach:** Onur Tezel <br>
**Datum:** 31/03/2026


| Naam  | Studentnummer |
|-------|---------------|
| Lucas | 25076116      |
| Sandro| 25154370      |
| Memhet| 25135007      |
| Julius| 25090216      |
| Dylan | 25118498      |
---

## **Inhoudsopgave**
1. [Imports & Configuratie](#1)
2. [Data inladen](#2)
3. [Data Exploration met SQL](#3)
4. [Onderzoeken Teameigenschappen](#4)

---
<a id='1'></a>
## **1. Imports & Configuratie**

In [21]:
# Imports
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Nieuwe visualisatie bibliotheken
import plotly.express as px
import nbformat

In [22]:
# Via deze functie in pandas zorgen we ervoor dat we de DataFrames volledig kunnen weergeven in de cell
pd.set_option('display.max_rows', 200)

---
<a id='2'></a>
## **2. Data inladen**
In de onderstaande kolommen laden we de data op dezelfde manier als de vorige keer, namelijk via de Kaggle API. <br> 
Hieronder vind je een uitleg over het gebruik en een referentie.



### **2.1 Gebruik van de Kaggle API** 
We gebruiken de Kaggle API om de data lokaal op te slaan en dit hoeft slechts eenmalig te gebeuren.<br>Download de API-sleutel uit onze GitHub-repository en plaats deze in de map C:\Users\<Jouw_gebruikersnaam>\.kaggle. <br>

Raadpleeg voor verdere instructies de officiële documentatie op https://www.kaggle.com/docs/api.<br>
Verwijder eventueel het # in het script, voer de pip install en de data-import uit en je bent klaar om de data lokaal te gebruiken.


In [23]:
# %pip install kaggle 
import kaggle 
kaggle.api.authenticate()
kaggle.api.dataset_download_files("hugomathien/soccer", path='.', unzip=True)

Dataset URL: https://www.kaggle.com/datasets/hugomathien/soccer


In [24]:
# Maak een verbinding met de SQLite-database "database.sqlite"
connection = sqlite3.connect("database.sqlite")

# Voer een SQL-query uit om alle gegevens uit de tabel 'Match' op te halen, het resultaat wordt ingelezen als pandas DataFrame
df = pd.read_sql("""
    
    SELECT *  
        FROM Match 
                 
    """, 
    connection)

---
<a id='3'></a>
## **3. Data Exploration met SQL**

### 3.1 Aantal gespeelde wedstrijden per seizoen

In [25]:
# Vaste variabelen voor Sprint 3
AZ_API_ID = 10229
EREDIVISIE_ID = 13274

def haal_wedstrijden_per_seizoen_op(team_id, conn):
    """
    Berekent het totaal aantal gespeelde wedstrijden per seizoen voor een specifiek team.
    (Vraag 1A)
    
    Parameters:
    - team_id (int): Het API ID van het team (bijv. 10229 voor AZ).
    - conn (sqlite3.Connection): De database verbinding.

    Returns:
    - pd.DataFrame: DataFrame met het seizoen en het aantal wedstrijden.
    """
    query  = """
    SELECT season, COUNT(*) AS aantal_gespeelde_wedstrijden
    FROM Match
    WHERE home_team_api_id = ? OR away_team_api_id = ?
    GROUP BY season
    ORDER BY season;
    """
    
    df = pd.read_sql(query, conn, params=(team_id, team_id))
    return df

# Uitvoeren:
df_1a = haal_wedstrijden_per_seizoen_op(AZ_API_ID, connection)
print(f'1A: AZ Alkmaar speelde in totaal {df_1a["aantal_gespeelde_wedstrijden"].sum()} wedstrijden.')
display(df_1a)

1A: AZ Alkmaar speelde in totaal 272 wedstrijden.


,season,aantal_gespeelde_wedstrijden
0,2008/2009,34
1,2009/2010,34
2,2010/2011,34
3,2011/2012,34
4,2012/2013,34
5,2013/2014,34
6,2014/2015,34
7,2015/2016,34


---
### 3.2 Aantal gespeelde wedstrijden kalender jaar 2010

In [26]:
def haal_wedstrijden_in_jaar_op(team_id, jaar, conn):
    """
    Berekent het aantal wedstrijden van een team in een specifiek kalenderjaar, verdeeld per seizoen.
    (Vraag 1B)
    
    Parameters:
    - team_id (int): Het API ID van het team.
    - jaar (str): Het kalenderjaar (bijv. '2010').
    - conn (sqlite3.Connection): De database verbinding.

    Returns:
    - pd.DataFrame: DataFrame met het seizoen en het aantal gespeelde wedstrijden.
    """
    query  = """
    SELECT season, COUNT(*) AS gespeelde_wedstrijden
    FROM Match
    WHERE (home_team_api_id = ? OR away_team_api_id = ?)
    AND strftime('%Y', date) = ? 
    GROUP BY season
    ORDER BY season;
    """
    
    df = pd.read_sql(query, conn, params=(team_id, team_id, str(jaar)))
    return df

# Uitvoeren:
df_1b = haal_wedstrijden_in_jaar_op(AZ_API_ID, '2010', connection)
print(f'1B: In het kalenderjaar 2010 speelde AZ {df_1b["gespeelde_wedstrijden"].sum()} wedstrijden.')
display(df_1b)

1B: In het kalenderjaar 2010 speelde AZ 34 wedstrijden.


,season,gespeelde_wedstrijden
0,2009/2010,16
1,2010/2011,18


---
### 3.3 Punten per seizoen & eindklassering

In [27]:
def haal_punten_per_team_op(league_id, conn):
    """
    Berekent het totaal aantal punten voor elk team per seizoen in een specifieke competitie.
    (Vraag 1C)
    
    Parameters:
    - league_id (int): Het ID van de competitie (bijv. 13274 voor de Eredivisie).
    - conn (sqlite3.Connection): De database verbinding.

    Returns:
    - pd.DataFrame: DataFrame met seizoen, teamnaam en totale punten.
    """
    query = """
    SELECT
        T.team_long_name AS Team,
        S.season,
        SUM(S.punten) AS totaal_punten
    FROM (
        SELECT season, home_team_api_id AS team_api_id,
            CASE WHEN home_team_goal > away_team_goal THEN 3
                 WHEN home_team_goal = away_team_goal THEN 1 ELSE 0 END AS punten
        FROM Match WHERE league_id = ?
        UNION ALL
        SELECT season, away_team_api_id AS team_api_id,
            CASE WHEN away_team_goal > home_team_goal THEN 3
                 WHEN away_team_goal = home_team_goal THEN 1 ELSE 0 END AS punten
        FROM Match WHERE league_id = ?
    ) AS S
    JOIN Team AS T ON S.team_api_id = T.team_api_id
    GROUP BY S.season, S.team_api_id, T.team_long_name
    ORDER BY S.season DESC, totaal_punten DESC;
    """
    
    df = pd.read_sql(query, conn, params=(league_id, league_id))
    df.index = df.index + 1 # Zorg dat de index bij 1 begint
    return df

# Uitvoeren:
df_1c = haal_punten_per_team_op(EREDIVISIE_ID, connection)
print("1C: Punten per team in de Eredivisie (eerste 10 getoond):")
display(df_1c.head(10))

1C: Punten per team in de Eredivisie (eerste 10 getoond):


,Team,season,totaal_punten
1,PSV,2015/2016,84
2,Ajax,2015/2016,82
3,Feyenoord,2015/2016,63
4,AZ,2015/2016,59
5,FC Utrecht,2015/2016,53
6,Heracles Almelo,2015/2016,51
7,FC Groningen,2015/2016,50
8,PEC Zwolle,2015/2016,48
9,Vitesse,2015/2016,46
10,N.E.C.,2015/2016,46


---
### 3.4 Eindpositie van AZ in de ranglijst

In [28]:
def haal_eindstand_team_op(team_id, league_id, conn):
    """
    Berekent de eindpositie van een specifiek team per seizoen met de SQL RANK() functie.
    (Vraag 1D)
    
    Parameters:
    - team_id (int): Het API ID van het team (AZ).
    - league_id (int): Het ID van de competitie (Eredivisie).
    - conn (sqlite3.Connection): De database verbinding.

    Returns:
    - pd.DataFrame: DataFrame met seizoen, totale punten en de eindpositie van het team.
    """
    query = """
    WITH PuntenPerTeam AS (
        SELECT S.season, S.team_api_id, SUM(S.punten) AS totaal_punten
        FROM (
            SELECT season, home_team_api_id AS team_api_id,
                CASE WHEN home_team_goal > away_team_goal THEN 3
                     WHEN home_team_goal = away_team_goal THEN 1 ELSE 0 END AS punten
            FROM Match WHERE league_id = ?
            UNION ALL
            SELECT season, away_team_api_id AS team_api_id,
                CASE WHEN away_team_goal > home_team_goal THEN 3
                     WHEN away_team_goal = home_team_goal THEN 1 ELSE 0 END AS punten
            FROM Match WHERE league_id = ?
        ) AS S
        GROUP BY S.season, S.team_api_id
    ),
    Ranglijst AS (
        SELECT season, team_api_id, totaal_punten,
               RANK() OVER(PARTITION BY season ORDER BY totaal_punten DESC) AS eindpositie
        FROM PuntenPerTeam
    )
    SELECT R.season AS Seizoen, T.team_long_name AS Team, R.totaal_punten AS Punten, R.eindpositie AS Eindstand
    FROM Ranglijst R
    JOIN Team T ON R.team_api_id = T.team_api_id
    WHERE R.team_api_id = ?
    ORDER BY R.season DESC;
    """
    
    df = pd.read_sql(query, conn, params=(league_id, league_id, team_id))
    df.index = df.index + 1
    return df

# Uitvoeren:
df_1d = haal_eindstand_team_op(AZ_API_ID, EREDIVISIE_ID, connection)
print("1D: Eindstand van AZ per seizoen in de Eredivisie:")
display(df_1d)

1D: Eindstand van AZ per seizoen in de Eredivisie:


,Seizoen,Team,Punten,Eindstand
1,2015/2016,AZ,59,4
2,2014/2015,AZ,62,3
3,2013/2014,AZ,47,8
4,2012/2013,AZ,39,10
5,2011/2012,AZ,65,4
6,2010/2011,AZ,59,4
7,2009/2010,AZ,62,5
8,2008/2009,AZ,80,1
